# 00 — Dicionário de dados

**CRISP-DM fase 2.2 — descrição dos dados** — Segmentação do mercado de veículos usados de Fortaleza

---

Este notebook monta o **dicionário de dados da base bruta**: para cada uma das
23 colunas de `data/raw/anuncios_detalhados.csv`, registra o nome da coluna,
uma descrição de negócio, a faixa (ou os extremos) dos valores observados e o
tipo de variável.

Aqui **não há download de uma base pública**: a camada bruta é o resultado direto do scraping da OLX
(ver `Software/coleta/`), já materializada em `data/raw/` e tratada como
imutável a partir deste ponto.

In [1]:
# importações
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 100)

from src import config

## Carregando a base de dados

A fonte é `data/raw/anuncios_detalhados.csv`: 2.565 anúncios únicos de
Fortaleza-CE, coletados por scraping da OLX e detalhados a 100% (ver
`Software/coleta/HISTORICO.md` para o histórico completo da coleta).

In [2]:
# Carregando base de dados
df = pd.read_csv(config.ARQUIVO_BRUTO)
origem = config.ARQUIVO_BRUTO

print(f"{origem.relative_to(RAIZ)} — {df.shape[0]:,} linhas × {df.shape[1]} colunas")
df.head()

data\raw\anuncios_detalhados.csv — 2,565 linhas × 23 colunas


,titulo,preco,localizacao,km,cor,motor,tipo,ano,link,valor_fipe_olx,marca,modelo,versao,cambio,combustivel,carroceria,portas,bairro,municipio,vendedor_tipo,aceita_troca,unico_dono,data_coleta
0,Chevrolet Corsa Sedan GLS 1.6 MPFI 4P 1999,11000,"Fortaleza, Jangurussu",100000,Prata,1.6,Sedã,1999.0,https://ce.olx.com.br/fortaleza-e-regiao/autos...,12320.0,Chevrolet,Corsa,Sedan GLS 1.6 MPFI 4P,Manual,Gasolina,Sedã,4 portas,Jangurussu,Fortaleza,particular,Sim,Não,2026-08-21
1,Volkswagen Saveiro 1.6 MI/ 1.6 MI Total Flex 8...,13599,"Fortaleza, Antônio Bezerra",109000,Prata,1.6,Pick-up,2009.0,https://ce.olx.com.br/fortaleza-e-regiao/autos...,NaN,Volkswagen,Saveiro,1.6 MI/ 1.6 MI Total Flex 8V,Manual,Flex,Pick-up,4 portas,Antônio Bezerra,Fortaleza,particular,Sim,Sim,2026-08-21
2,Volkswagen Fox City 1.0 MI/ 1.0mi Total Flex 8...,15000,"Fortaleza, Passaré",196000,Preto,1.0,Hatch,2006.0,https://ce.olx.com.br/fortaleza-e-regiao/autos...,NaN,Volkswagen,Fox,City 1.0 MI/ 1.0mi Total Flex 8V 5P,Manual,Flex,Hatch,4 portas,Passaré,Fortaleza,particular,Não,Não,2026-08-21
3,Ford KA 1.0i 3P 2007,9500,"Fortaleza, Passaré",1600,Preto,1.0,Hatch,2007.0,https://ce.olx.com.br/fortaleza-e-regiao/autos...,13757.0,Ford,KA,1.0i 3P,Manual,Gasolina,Hatch,2 portas,Passaré,Fortaleza,particular,NaN,Não,2026-08-21
4,Ford KA 1.0i 3P 2007,8000,"Fortaleza, Passaré",160000,Preto,1.0,Hatch,2007.0,https://ce.olx.com.br/fortaleza-e-regiao/autos...,13757.0,Ford,KA,1.0i 3P,Manual,Gasolina,Hatch,2 portas,Passaré,Fortaleza,particular,NaN,Não,2026-08-21


## Varredura das colunas

Percorremos coluna a coluna classificando o tipo e coletando os valores de
referência:

| dtype | tipo atribuído | valores registrados |
|---|---|---|
| inteiro | Discreta | mínimo e máximo |
| decimal | Contínua | mínimo e máximo |
| texto | Nominal | quantidade de categorias e exemplos |

Esta base **não tem colunas com escala
ordinal declarada** (tipo `Ex > Gd > TA > Fa > Po`) — todas as colunas de
texto são tratadas como Nominais nesta etapa.

In [3]:
# lista de dicionários para armazenar informações sobre as colunas
col_info = []


# Converte escalar numpy em escalar Python. Sem isso o CSV sai com
# `[np.int64(12789), np.int64(755000)]` em vez de `[12789, 755000]` — é só o
# `repr` do numpy 2, mas o dicionário é lido por gente, não por máquina.
def escalar(valor):
    return valor.item() if hasattr(valor, "item") else valor


# percorrendo as colunas do dataframe
for col in df.columns:
    serie = df[col]
    n_nulos = int(serie.isna().sum())

    if pd.api.types.is_integer_dtype(serie):
        tipo = "Discreta"
        valores = [escalar(serie.min()), escalar(serie.max())]
    elif pd.api.types.is_float_dtype(serie):
        tipo = "Contínua"
        valores = [escalar(serie.min()), escalar(serie.max())]
    else:
        tipo = "Nominal"
        categorias = sorted(serie.dropna().unique().tolist())
        valores = categorias if len(categorias) <= 10 else categorias[:10] + ["…"]

    col_info.append(
        {
            "Variável": col,
            "Tipo": tipo,
            "Valores/Faixa": valores,
            "Nulos": n_nulos,
            "% nulos": round(100 * n_nulos / len(df), 1),
        }
    )

In [4]:
# transformando lista de dicionários em dataframe
col_info_df = pd.DataFrame(col_info)

print(f"{len(col_info_df)} colunas descritas")
col_info_df["Tipo"].value_counts()

23 colunas descritas


Tipo
Nominal     19
Discreta     2
Contínua     2
Name: count, dtype: int64

## Descrições de negócio

Cada coluna recebe uma descrição em linguagem de negócio, casada por nome —
não por posição — para não quebrar em silêncio se a ordem das colunas mudar.

In [5]:
# Descrições de negócio, por nome da coluna
DESCRICOES = {
    "titulo": "Título do anúncio, como publicado pelo vendedor na OLX",
    "preco": "Preço anunciado, em reais (R$)",
    "localizacao": "Localização declarada no card de listagem (cidade, bairro)",
    "km": "Quilometragem informada no anúncio",
    "cor": "Cor do veículo",
    "motor": "Motorização declarada no card de listagem (ex.: '1.6')",
    "tipo": "Tipo/carroceria conforme o card de listagem",
    "ano": "Ano do veículo, extraído do título do anúncio",
    "link": "URL do anúncio na OLX (chave única do anúncio)",
    "valor_fipe_olx": "Valor de referência FIPE mostrado pela própria OLX no anúncio, quando disponível",
    "marca": "Marca do veículo, extraída do dataLayer da página do anúncio",
    "modelo": "Modelo do veículo (dataLayer)",
    "versao": "Versão/acabamento do veículo (dataLayer)",
    "cambio": "Tipo de câmbio (Manual, Automático, Semi-Automático, Automatizado)",
    "combustivel": "Tipo de combustível (Gasolina, Flex, Álcool, Diesel, Híbrido, Elétrico)",
    "carroceria": "Carroceria do veículo, conforme o dataLayer (Sedã, Hatch, SUV, Pick-up…)",
    "portas": "Número de portas do veículo",
    "bairro": "Bairro onde o anunciante está localizado",
    "municipio": "Município onde o anunciante está localizado",
    "vendedor_tipo": "Tipo de vendedor: 'particular' ou 'profissional' (loja/revenda)",
    "aceita_troca": "Indica se o vendedor aceita troca por outro veículo (Sim/Não)",
    "unico_dono": "Indica se o veículo é de único dono (Sim/Não)",
    "data_coleta": "Data em que o anúncio foi coletado (scraping)",
}

len(DESCRICOES)

23

In [6]:
# Adicionando descrição para cada uma das variáveis, casada por nome de coluna
col_info_df["Descrição"] = col_info_df["Variável"].map(DESCRICOES)

faltando = col_info_df[col_info_df["Descrição"].isna()]
assert faltando.empty, f"Colunas sem descrição: {faltando['Variável'].tolist()}"

col_info_df

,Variável,Tipo,Valores/Faixa,Nulos,% nulos,Descrição
0,titulo,Nominal,"[Asia Motors Towner Coach Full 1995, Asia Moto...",0,0.0,"Título do anúncio, como publicado pelo vendedo..."
1,preco,Discreta,"[5000, 250000]",0,0.0,"Preço anunciado, em reais (R$)"
2,localizacao,Nominal,"[Acarape, Aquiraz, Aquiraz, Tapera, Aracoiaba,...",0,0.0,Localização declarada no card de listagem (cid...
3,km,Discreta,"[0, 999998]",0,0.0,Quilometragem informada no anúncio
4,cor,Nominal,"[Amarelo, Azul, Branco, Cinza, Laranja, Prata,...",78,3.0,Cor do veículo
5,motor,Nominal,"[1.0, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, ...",69,2.7,Motorização declarada no card de listagem (ex....
6,tipo,Nominal,"[Buggy, Caminhão Leve, Conversível, Coupé, Hat...",165,6.4,Tipo/carroceria conforme o card de listagem
7,ano,Contínua,"[1972.0, 2027.0]",1,0.0,"Ano do veículo, extraído do título do anúncio"
8,link,Nominal,[https://ce.olx.com.br/fortaleza-e-regiao/auto...,0,0.0,URL do anúncio na OLX (chave única do anúncio)
9,valor_fipe_olx,Contínua,"[3180.0, 322821.0]",238,9.3,Valor de referência FIPE mostrado pela própria...


## Salvando o dicionário de dados

O arquivo vai para `data/processed/dicionario.csv`, separado por `;` e sem
índice — o mesmo formato consumido pelos demais notebooks.

> As descrições da **camada curada** (colunas derivadas/agregadas na etapa de
> preparação) vivem em `src/data/dicionario.py`. Este CSV descreve a base
> **bruta**: é o retrato de entrada, anterior a qualquer transformação.

In [7]:
# Salvando dicionário de dados
config.garantir_diretorios()
destino = config.DATA_PROCESSED / "dicionario.csv"
col_info_df.to_csv(destino, index=False, sep=";")

print(f"Dicionário gravado em {destino.relative_to(RAIZ)} ({destino.stat().st_size:,} bytes)")
pd.read_csv(destino, sep=";").head(10)

Dicionário gravado em data\processed\dicionario.csv (5,271 bytes)


,Variável,Tipo,Valores/Faixa,Nulos,% nulos,Descrição
0,titulo,Nominal,"['Asia Motors Towner Coach Full 1995', 'Asia M...",0,0.0,"Título do anúncio, como publicado pelo vendedo..."
1,preco,Discreta,"[5000, 250000]",0,0.0,"Preço anunciado, em reais (R$)"
2,localizacao,Nominal,"['Acarape', 'Aquiraz', 'Aquiraz, Tapera', 'Ara...",0,0.0,Localização declarada no card de listagem (cid...
3,km,Discreta,"[0, 999998]",0,0.0,Quilometragem informada no anúncio
4,cor,Nominal,"['Amarelo', 'Azul', 'Branco', 'Cinza', 'Laranj...",78,3.0,Cor do veículo
5,motor,Nominal,"['1.0', '1.2', '1.3', '1.4', '1.5', '1.6', '1....",69,2.7,Motorização declarada no card de listagem (ex....
6,tipo,Nominal,"['Buggy', 'Caminhão Leve', 'Conversível', 'Cou...",165,6.4,Tipo/carroceria conforme o card de listagem
7,ano,Contínua,"[1972.0, 2027.0]",1,0.0,"Ano do veículo, extraído do título do anúncio"
8,link,Nominal,['https://ce.olx.com.br/fortaleza-e-regiao/aut...,0,0.0,URL do anúncio na OLX (chave única do anúncio)
9,valor_fipe_olx,Contínua,"[3180.0, 322821.0]",238,9.3,Valor de referência FIPE mostrado pela própria...
